# 3 · Data Profiling (Bronze)
Profile all 6 Bronze tables: counts, duplicates, nulls, extreme values, categories, keys and relationships.
Results feed the Day 2 data-quality rules.

In [0]:

from pyspark.sql import functions as F

spark.sql("USE capstone_5iot.iot_schema")

# table -> key
TABLES = {
    "bronze_sensor_readings":     "reading_id",
    "bronze_failure_events":      "failure_event_id",
    "bronze_work_orders":         "work_order_id",
    "bronze_machine_master":      "machine_id",
    "bronze_plant_master":        "plant_id",
    "bronze_failure_code_lookup": "failure_code",
}
def data_cols(df):
    return [c for c in df.columns if not c.startswith("_")]

sensor = spark.table("bronze_sensor_readings")

## 1. Row counts, keys and duplicates

In [0]:
rows = []
for t, key in TABLES.items():
    df = spark.table(t)
    n  = df.count()
    rows.append((t, n, df.select(key).distinct().count(), n - df.select(data_cols(df)).distinct().count()))

display(spark.createDataFrame(rows, "table string, rows long, distinct_keys long, exact_duplicates long"))

## 2. Null counts

In [0]:
rows = []
for t in TABLES:
    df = spark.table(t)
    nulls = df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in data_cols(df)]).first().asDict()
    rows += [(t, c, n) for c, n in nulls.items() if n > 0]

display(spark.createDataFrame(rows, "table string, column string, nulls long"))

## 3. Sensor metrics – min / max / mean

In [0]:
METRICS = ["temperature_c", "vibration_mm_s", "pressure_bar", "rpm"]
num = sensor.select([F.col(c).cast("double").alias(c) for c in METRICS])
display(num.summary("count", "min", "25%", "50%", "75%", "max", "mean"))

## 4. Extreme values (temperature outside -50 to 200 °C)

In [0]:
%sql
SELECT temperature_c, COUNT(*) AS readings
FROM bronze_sensor_readings
WHERE CAST(temperature_c AS DOUBLE) NOT BETWEEN -50 AND 200
GROUP BY temperature_c

## 5. Category values

In [0]:
CATS = {
    "bronze_sensor_readings": ["operating_mode", "sensor_quality", "firmware_version"],
    "bronze_failure_events":  ["failure_code", "severity", "reported_by", "event_status"],
    "bronze_work_orders":     ["maintenance_type", "work_order_status"],
    "bronze_machine_master":  ["machine_type", "criticality", "operating_status"],
}
rows = []
for t, cols in CATS.items():
    for c in cols:
        for r in spark.table(t).groupBy(c).count().collect():
            rows.append((t, c, r[c], r["count"]))

display(spark.createDataFrame(rows, "table string, column string, value string, count long"))

## 6. Time ranges

In [0]:
%sql
SELECT 'sensor readings' AS feed, MIN(reading_ts) AS first_ts, MAX(reading_ts) AS last_ts FROM bronze_sensor_readings
UNION ALL SELECT 'failure events', MIN(failure_ts), MAX(failure_ts) FROM bronze_failure_events
UNION ALL SELECT 'work orders', MIN(created_ts), MAX(created_ts) FROM bronze_work_orders

## 7. Relationships (orphan keys)

In [0]:
FK = [
    ("bronze_sensor_readings", "machine_id",    "bronze_machine_master",      "machine_id"),
    ("bronze_sensor_readings", "plant_id",      "bronze_plant_master",        "plant_id"),
    ("bronze_machine_master",  "plant_id",      "bronze_plant_master",        "plant_id"),
    ("bronze_failure_events",  "machine_id",    "bronze_machine_master",      "machine_id"),
    ("bronze_failure_events",  "work_order_id", "bronze_work_orders",         "work_order_id"),
    ("bronze_failure_events",  "failure_code",  "bronze_failure_code_lookup", "failure_code"),
    ("bronze_work_orders",     "machine_id",    "bronze_machine_master",      "machine_id"),
]
fk_rows = []
for child, ck, parent, pk in FK:
    keys    = spark.table(parent).select(F.col(pk).alias(ck)).distinct()
    orphans = spark.table(child).join(keys, ck, "left_anti").count()
    fk_rows.append((f"{child}.{ck} -> {parent}.{pk}", orphans))

display(spark.createDataFrame(fk_rows, "relationship string, orphans long")) 

## 8. Business anomalies (input for Day 2 DQ rules)

In [0]:
%sql
SELECT 'readings before machine commissioned' AS check, COUNT(*) AS rows
FROM bronze_sensor_readings s JOIN bronze_machine_master m USING (machine_id)
WHERE to_date(s.reading_ts) < CAST(m.commissioned_date AS DATE)
UNION ALL
SELECT 'readings from INACTIVE machines', COUNT(*)
FROM bronze_sensor_readings s JOIN bronze_machine_master m USING (machine_id)
WHERE m.operating_status = 'INACTIVE'
UNION ALL
SELECT 'work orders not COMPLETED but have completed_ts', COUNT(*)
FROM bronze_work_orders
WHERE work_order_status <> 'COMPLETED' AND completed_ts IS NOT NULL

## 9. Validation – profile matches the expected source profile
| Check | Expected |
|---|---|
| Sensor rows / unique reading_id / exact duplicates | 11,653 / 11,612 / 41 |
| Sensor nulls (pressure 5, firmware 5, quality 4, mode 1) | 15 |
| Extreme temperatures (999999.99 / -999) | 46 |
| Unparseable timestamps | 0 |
| Orphan keys | 0 |

In [0]:
cols = data_cols(sensor)
ts   = sensor.select(F.to_timestamp("reading_ts").alias("r"), F.to_timestamp("received_ts").alias("c"))

actual = {
    "sensor rows":            sensor.count(),
    "unique reading_id":      sensor.select("reading_id").distinct().count(),
    "exact duplicates":       sensor.count() - sensor.select(cols).distinct().count(),
    "sensor nulls":           sum(sensor.filter(F.col(c).isNull()).count() for c in cols),
    "extreme temperatures":   sensor.filter(~F.col("temperature_c").cast("double").between(-50, 200)).count(),
    "unparseable timestamps": ts.filter("r IS NULL OR c IS NULL").count(),
    "orphan keys":            sum(r[1] for r in fk_rows),
}
expected = {"sensor rows": 11653, "unique reading_id": 11612, "exact duplicates": 41, "sensor nulls": 15,
            "extreme temperatures": 46, "unparseable timestamps": 0, "orphan keys": 0}

res = spark.createDataFrame([(k, expected[k], actual[k], "PASS" if expected[k] == actual[k] else "FAIL") for k in expected],
                            "check string, expected long, actual long, status string")
display(res)
assert res.filter("status = 'FAIL'").count() == 0, "Profile does not match expected"